# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates end-to-end exploration and processing of a biomedical dataset defined by a Croissant schema using the `mlcroissant` library. You will learn how to load metadata, explore record sets and fields (referenced by their `@id`), extract data, perform common data transformations, and visualize relationships.

### Dataset Source
The dataset schema is provided via Croissant at:
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed. If running locally or in Colab, uncomment:
!pip install -U mlcroissant pandas matplotlib

## 1. Data Loading
Load dataset metadata and records with `mlcroissant`. We'll inspect the dataset's title and summary after loading.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata (access as an object, not a dict)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Examine available record sets, fields, and their `@id` values. This will guide subsequent data extraction steps.

**Note:** All identifiers used will reference each schema component by its unique `@id`.

In [ ]:
# List all record sets and their fields by @id
print("Record Sets in dataset:")

record_sets = dataset.metadata.recordSets
record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        # For each field, show name and @id
        print(f"    - {field.name} (@id: {field.id})")
    print("")

## 3. Data Extraction
We will extract all available record sets by referencing their `@id` values and store each as a pandas DataFrame for further analysis.

Each DataFrame's columns are referenced by their field `@id`, so we can work with the correct identifiers programmatically.

In [ ]:
# Extract all record sets into DataFrames using their @id
dataframes = {}
# Use discovered record set @ids from above (if none, dataset may not expose public records for all, consult Croissant schema)
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"   No data records found for {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"   Columns (@id): {df.columns.tolist()}")
    print(df.head(2), "\n")

# Select first non-empty record set for downstream examples
chosen_record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        chosen_record_set_id = rs_id
        break
if chosen_record_set_id is None:
    raise ValueError("No record sets with data available!")
else:
    print(f"Using record set '{chosen_record_set_id}' for EDA below.")
    print("Columns:", dataframes[chosen_record_set_id].columns.tolist())
    display(dataframes[chosen_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
In this section, we:
- Select a numeric field by its `@id`.
- Filter records for values above a given threshold.
- Normalize that field.
- Group and summarize by a categorical (text) field.

Replace `numeric_field_id` and `group_field_id` with the appropriate `@id` values printed above for the record set.

In [ ]:
# Choose a numeric and a group field by @id (as displayed above)
# Example: suppose 'age_at_diagnosis' and 'sex' are present as field @ids in the dataset
numeric_field_id = None
group_field_id = None

# Try to auto-select some likely candidates for demonstration:
df = dataframes[chosen_record_set_id]
# Find first numeric-looking column
for col in df.columns:
    # Try to convert to numeric; skip on error
    try:
        if pd.api.types.is_numeric_dtype(df[col]) or pd.to_numeric(df[col], errors='coerce').notnull().sum() > 0:
            numeric_field_id = col
            break
    except Exception:
        continue
# Pick a non-numeric field for grouping
for col in df.columns:
    # Look for simple object/string type, not same as numeric field
    if col != numeric_field_id and (df[col].dtype == "object" or df[col].dtype.name == "category"):
        group_field_id = col
        break

if numeric_field_id is None:
    raise ValueError("No numeric field found to use for filtering and normalization.")

print(f"Using numeric_field_id: {numeric_field_id}")
print(f"Using group_field_id: {group_field_id}")

# Coerce to numeric for filtering/processing
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
# Remove rows where numeric is NaN
filtered_df = df[df[numeric_field_id].notnull()]

threshold = filtered_df[numeric_field_id].mean()  # Demo: use mean as threshold
filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered rows with {numeric_field_id} > {threshold:.1f}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field and calculate mean of the numeric
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the numeric variable and compare distributions by a group field if available.

In [ ]:
if numeric_field_id:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].plot.hist(bins=15, alpha=0.7, color='steelblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

if group_field_id and numeric_field_id:
    plt.figure(figsize=(8, 6))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
We've demonstrated:
- Programmatic exploration of a FAIR biomedical dataset using `mlcroissant`
- Navigating the schema using `@id` references for record sets and fields
- Extracting and working with tabular records in pandas
- Filtering, normalization, grouping, and plotting using schema-driven identifiers

This approach ensures repeatable, future-proof workflows on any Croissant-compatible dataset.

_For more details, consult the [mlcroissant documentation](https://mlcommons.org/croissant)._